torch.nn<br>
├── nn.Module          ← base class for ALL models, you'll subclass this <br>
├── nn.Linear          ← fully connected layer (y = xW + b)<br>
├── Activation fns     ← nn.ReLU, nn.Sigmoid, nn.Softmax<br>
├── Loss fns           ← nn.MSELoss, nn.CrossEntropyLoss<br>
├── nn.Sequential      ← quick way to stack layers<br>
├── Optimizers         ← torch.optim.SGD, Adam (not nn but used with it)<br>
└── Training loop      ← forward → loss → backward → update<br>

Order :
1. nn.Module + nn.Linear    ← what a layer actually is
2. Activations              ← why they exist, which to use when
3. Loss functions           ← MSE vs CrossEntropy, when to use each
4. nn.Sequential            ← building simple models fast
5. Optimizers               ← SGD vs Adam
6. Full training loop       ← putting it all together

In [ ]:
import sklearn
from sklearn.datasets import make_circles
import pandas as pd
import matplotlib.pyplot as plt

#number of samples
n_smaples = 1000

X, y = make_circles(n_samples=n_smaples,noise=0.03,random_state=42)
print(f"X:{X[:5]}")
print(f"y:{y[:5]}")

circles = pd.DataFrame({"x1":X[:,0],"x2":X[:,1],"label":y})
print(circles.head(10))

plt.scatter(X[:,0],X[:,1],c=y,cmap=plt.cm.RdBu)
plt.show()





In [ ]:
from sklearn.model_selection import train_test_split
import torch

X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)


X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

print(len(X_train), len(y_train))
print(len(X_test), len(y_test))

print(X_train.shape)
print(X_test.shape)



def plot_map():
    plt.scatter(X_train[:,0],X_train[:,1],c="blue",s=4,label="train_data")
    plt.scatter(X_test[:,0],X_test[:,1],c="green",s=4,label="test_data")
    plt.legend()
    plt.show()

plot_map()

In [ ]:
#create model
import torch
from torch import nn

class circleModel0(nn.Module):
    def __init__(self):
        super().__init__()
        self.linearLayers = nn.Sequential(
            nn.Linear(in_features=2,out_features=50),
            nn.ReLU(),
            nn.Linear(in_features=50,out_features=10),
            nn.ReLU(),
            nn.Linear(in_features=10,out_features=1)
            
            # nn.ReLU(),
            # nn.Linear(in_features=20,out_features=10),
            # nn.ReLU(),
            # nn.Linear(in_features=10,out_features=1)
        )

    def forward(self,X):
        return self.linearLayers(X)
    

model_0 = circleModel0()
print(model_0.state_dict())
# print(model_0.children())
# print(model_0.named_parameters())
# print(model_0.parameters())


In [ ]:
# loss and optimize objects
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(params=model_0.parameters(),lr=0.5)

#train and test loop

In [ ]:
def accuracy(x_actual,x_pred):
    correct = torch.eq(x_actual,x_pred).sum().item()
    accuracy = correct/(len(x_actual))*100
    # print(accuracy)
    return accuracy

# accuracy(torch.tensor([1,2,3,4]),torch.tensor([1,4,3,1]))

In [ ]:
epoches = 500
train_loss = []
test_loss = []
epoch_cnt = []

for epoch in range(epoches):

    model_0.train()
    y_tr_logit = model_0(X_train).squeeze()
    y_tr_prob = torch.sigmoid(y_tr_logit)
    y_tr_label = torch.round(y_tr_prob).squeeze()

    # print("logit:",y_tr_logit.shape,X_train.shape)
    loss = loss_fn(y_tr_logit,y_train)
    acc_train = accuracy(y_train,y_tr_label)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model_0.eval()
    with torch.no_grad():
        y_test_logit = model_0(X_test).squeeze()
        y_test_prob = torch.sigmoid(y_test_logit)
        y_test_label = torch.round(y_test_prob).squeeze()

        loss_test = loss_fn(y_test_logit,y_test)
        acc_test = accuracy(y_test,y_test_label)

    if epoch % 10 ==  0:
        train_loss.append(loss)
        test_loss.append(loss_test)
        epoch_cnt.append(epoch)
        print(f"epoch:{epoch},Train_loss:{loss},Test_loss:{loss_test},Accuracy_train:{acc_train},Accuracy_test:{acc_test}")


In [ ]:
import requests
from pathlib import Path

#https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/refs/heads/main/helper_functions.py
if Path("helper_functions.py").is_file():
    print("File exists")
else:
    request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/refs/heads/main/helper_functions.py")
    with open("helper_functions.py","wb") as f:
        f.write(request.content)

from helper_functions import plot_decision_boundary

In [ ]:
plt.Figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.title("Training data")
plot_decision_boundary(model_0,X_train,y_train)
plt.subplot(1,2,2)
plt.title("Test data")
plot_decision_boundary(model_0,X_test,y_test)


In [ ]:
# loss curve
import numpy as np

train_loss = np.array(torch.tensor(train_loss).numpy())
test_loss = np.array(torch.tensor(test_loss).numpy())

plt.plot(epoch_cnt,train_loss,label="train_loss")
plt.plot(epoch_cnt,test_loss,label="test_loss")
plt.xlabel("Epoch")
plt.ylabel("loss")
plt.title("Loss curves")
plt.legend()
plt.show()

train loss down, test loss down together  → healthy, keep training<br >
train loss down, test loss UP             → overfitting, stop here<br >
both losses flat and high                 → model not learning, check lr/architecture<br >
loss spikes                               → lr too high<br >
loss never moves                          → lr too low or wrong loss function<br >

The 4 things nn.Module gives you:<br >

# 1. Parameter tracking — finds all weights automatically
model.parameters()        # all weights and biases
model.state_dict()        # all parameters as a dict

# 2. Device movement — moves ALL layers at once
model.to('cuda')          # moves every layer together

# 3. Train/eval switching
model.train()             # enables dropout, batchnorm etc
model.eval()              # disables them for inference

# 4. Save/load
torch.save(model.state_dict(), 'model.pth')
model.load_state_dict(torch.load('model.pth'))



# Sequential — layers always go in fixed order
# Manual forward() — you can do anything:<br>

def forward(self, x): <br>
    x1 = self.layer1(x)<br>
    x2 = self.layer2(x)<br>
    return x1 + x2        # skip connections (ResNet does this)<br>

def forward(self, x1, x2):<br>
    return self.layer1(x1) + self.layer2(x2)   # multiple inputs<br>

nn.Linear — what's actually inside: <br>
pythonlayer = nn.Linear(in_features=2, out_features=4)<br>

# internally holds:<br>
print(layer.weight.shape)   # [4, 2]  ← W matrix<br>
print(layer.bias.shape) # [4]     ← b vector <br>

In [ ]:
# save model and load

torch.save(model_0.state_dict(),"model.pth")
model_0.load_state_dict(torch.load("model.pth"))

state_dict         = just the weights (numbers)<br>
model architecture = the structure (layers, connections)<br>

torch.save  → saves weights only, NOT the structure<br>
torch.load  → loads weights back INTO an existing structure<br>

In [ ]:
model_0.state_dict()

# total number of parameters


In [ ]:
param_cnt = sum(p.numel() for p in model_0.parameters())
param_cnt

# Activations

1. Which activation for which situation:<br>
Hidden layers  → ReLU        (default choice, fast, works well)<br>
Binary output  → nothing*    (BCEWithLogitsLoss handles sigmoid internally)<br>
Multi-class    → nothing*    (CrossEntropyLoss handles softmax internally)<br>
Regression     → nothing     (raw number output, no activation)<br>

* applying sigmoid/softmax manually + BCE/CrossEntropy = numerical instability<br>
  PyTorch combines them internally for stability — that's why your model<br>
  has no activation on the final layer<br>
2. Why ReLU specifically:<br>
pythonReLU(x) = max(0, x)   # negative → 0, positive → unchanged<br>

# Without activation: stacking linear layers = still just one linear layer
# y = W2(W1x) = (W2W1)x   ← just one matrix multiply, pointless stacking

# With ReLU: non-linearity breaks this — layers can learn complex patterns
# circles dataset needs non-linearity — inner vs outer can't be separated
# by a straight line<br>
3. The ones you'll encounter:<br>
ReLU      → hidden layers, most common<br>
LeakyReLU → fixes "dying ReLU" problem (neurons stuck at 0)<br>
Sigmoid   → binary output manually (avoid with BCE)<br>
Softmax   → multiclass output manually (avoid with CrossEntropy)<br>
Tanh      → RNNs, sometimes hidden layers<br>

# Loss
1. MSELoss          → Regression    (predicting a number)
2. BCEWithLogitsLoss → Binary classification   (yes/no, 0/1)
3. CrossEntropyLoss  → Multi-class classification (cat/dog/bird)

# nn.Sequential

1. Naming layers — easier debugging
    Named way — more readable in state_dict and debugging from collections import OrderedDict

    nn.Sequential(OrderedDict([
        ('layer1', nn.Linear(2, 50)),
        ('relu1',  nn.ReLU()),
        ('layer2', nn.Linear(50, 10))
    ]))
2. When Sequential is NOT enough:

straight pipeline         → Sequential, clean and simple<br>
any custom flow           → nn.Module with manual forward()<br>
real production models    → almost always manual forward()<br>

# Optimizers

Loss landscape = hilly terrain, trying to reach the valley (minimum)

SGD  → walks downhill, fixed step size
       can get stuck, slow on flat areas

Adam → walks downhill, adjusts step size per direction
       faster, handles flat/steep areas better
====================

Starting a new problem        → Adam, lr=0.001 (safe default)<br>
Not converging                → try lower lr<br>
Loss spiking                  → lr too high, reduce it<br>
Research/fine tuning          → SGD with momentum, more control<br>
Transformers (BERT, GPT)      → AdamW (Adam with weight decay)<br>



model_0.train()       &nbsp;&nbsp;  # tells layers like Dropout "we are training" <br>
model_0.eval()         &nbsp;&nbsp;   # tells layers like Dropout "we are evaluating"<br>
                       &nbsp;&nbsp;  # your model has no dropout but good habit always<br>
<br>
optimizer.zero_grad() &nbsp;&nbsp; # gradients accumulate by default<br>
                      &nbsp;&nbsp; # MUST clear before each backward()<br>
<br>
loss.backward()       &nbsp;&nbsp; # computes d(loss)/d(every weight)<br>
                      &nbsp;&nbsp; # walks computation graph backwards<br>
<br>
optimizer.step()      &nbsp;&nbsp; # weight = weight - lr * gradient<br>
                      &nbsp;&nbsp; # actually moves the weights<br>
<br>
torch.no_grad()       &nbsp;&nbsp; # eval needs no gradients<br>
                      &nbsp;&nbsp; # no graph built → faster + less memory<br>
<br>
loss.item()           &nbsp;&nbsp; # converts tensor(0.432) → 0.432<br>
                      &nbsp;&nbsp; # detaches from graph, safe to store in list<br>